# Step 2: Train the Tiny 1D CNN
**Target: PYNQ-Z2 (Zynq-7020, 220 DSPs)**

Architecture: 3-layer 1D CNN → ~1,500 parameters → fits in BRAM easily.

Run Step 1 first to generate the `.npy` feature files.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_val = np.load('X_val.npy')
y_val = np.load('y_val.npy')
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


In [ ]:
# === MODEL DEFINITION ===
class TinyCNN1D(nn.Module):
    """
    Tiny 1D CNN designed to fit on Zynq-7020 after INT8 quantization.
    
    Resource budget:
      Conv1: 1*8*5  = 40 MACs per position  -> ~8 DSPs
      Conv2: 8*16*3 = 384 MACs per position  -> ~24 DSPs
      Conv3: 16*16*3= 768 MACs per position  -> ~16 DSPs (reuse)
      FC:    64*4   = 256 MACs               -> minimal
      TOTAL: ~48 DSPs (of 220 available)
    """
    def __init__(self, num_classes=4):
        super().__init__()
        # Input: (batch, 1, 64)
        self.conv1 = nn.Conv1d(1, 8, kernel_size=5, padding=2)    # -> (B, 8, 64)
        self.bn1 = nn.BatchNorm1d(8)
        self.pool1 = nn.MaxPool1d(2)                               # -> (B, 8, 32)
        
        self.conv2 = nn.Conv1d(8, 16, kernel_size=3, padding=1)   # -> (B, 16, 32)
        self.bn2 = nn.BatchNorm1d(16)
        self.pool2 = nn.MaxPool1d(2)                               # -> (B, 16, 16)
        
        self.conv3 = nn.Conv1d(16, 16, kernel_size=3, padding=1)  # -> (B, 16, 16)
        self.bn3 = nn.BatchNorm1d(16)
        self.pool3 = nn.AdaptiveAvgPool1d(4)                       # -> (B, 16, 4)
        
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(16 * 4, num_classes)                   # 64 -> 4
    
    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool3(torch.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = TinyCNN1D(num_classes=4).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")
print(f"\nModel architecture:")
print(model)


In [ ]:
# === PREPARE DATA LOADERS ===
# Normalize input to [0, 1]
X_train_norm = X_train / 255.0
X_val_norm = X_val / 255.0
X_test_norm = X_test / 255.0

# Convert to tensors, add channel dim: (N,) -> (N, 1, 64)
X_train_t = torch.from_numpy(X_train_norm).float().unsqueeze(1).to(device)
y_train_t = torch.from_numpy(y_train).long().to(device)
X_val_t = torch.from_numpy(X_val_norm).float().unsqueeze(1).to(device)
y_val_t = torch.from_numpy(y_val).long().to(device)
X_test_t = torch.from_numpy(X_test_norm).float().unsqueeze(1).to(device)
y_test_t = torch.from_numpy(y_test).long().to(device)

BATCH_SIZE = 256
train_ds = TensorDataset(X_train_t, y_train_t)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_ds = TensorDataset(X_val_t, y_val_t)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_ds = TensorDataset(X_test_t, y_test_t)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE)

print(f"Batches per epoch: train={len(train_dl)}, val={len(val_dl)}, test={len(test_dl)}")


In [ ]:
# === TRAINING ===
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

EPOCHS = 40
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9} | {'LR':>8}")
print("-" * 70)

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
        train_correct += (out.argmax(1) == yb).sum().item()
        train_total += xb.size(0)
    
    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * xb.size(0)
            val_correct += (out.argmax(1) == yb).sum().item()
            val_total += xb.size(0)
    
    t_loss = train_loss / train_total
    t_acc = 100 * train_correct / train_total
    v_loss = val_loss / val_total
    v_acc = 100 * val_correct / val_total
    lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    
    scheduler.step(v_loss)
    
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), 'best_model.pth')
        marker = ' ★'
    else:
        marker = ''
    
    print(f"{epoch+1:5d} | {t_loss:10.4f} | {t_acc:8.2f}% | {v_loss:10.4f} | {v_acc:8.2f}%{marker} | {lr:.1e}")

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")


In [ ]:
# === TRAINING CURVES (for YouTube video) ===
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0a0a1a')
    ax.tick_params(colors='white')
    ax.spines['bottom'].set_color('#333')
    ax.spines['left'].set_color('#333')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

ax1.plot(history['train_loss'], color='#e94560', linewidth=2, label='Train')
ax1.plot(history['val_loss'], color='#16c79a', linewidth=2, label='Val')
ax1.set_title('Loss', color='white', fontsize=14)
ax1.set_xlabel('Epoch', color='white')
ax1.legend(facecolor='#1a1a2e', edgecolor='#333', labelcolor='white')

ax2.plot(history['train_acc'], color='#e94560', linewidth=2, label='Train')
ax2.plot(history['val_acc'], color='#16c79a', linewidth=2, label='Val')
ax2.set_title('Accuracy (%)', color='white', fontsize=14)
ax2.set_xlabel('Epoch', color='white')
ax2.legend(facecolor='#1a1a2e', edgecolor='#333', labelcolor='white')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, facecolor='#1a1a2e', bbox_inches='tight')
plt.show()


In [ ]:
# === TEST SET EVALUATION ===
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_preds = []
all_true = []

with torch.no_grad():
    for xb, yb in test_dl:
        out = model(xb)
        preds = out.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

all_preds = np.array(all_preds)
all_true = np.array(all_true)

test_acc = 100 * (all_preds == all_true).mean()
print(f"Test Accuracy: {test_acc:.2f}%")

# Per-class accuracy
CLASS_NAMES = ['Drone', 'Bird', 'Human', 'Corner Reflector']
print(f"\nPer-class accuracy:")
for c in range(4):
    mask = all_true == c
    if mask.sum() > 0:
        acc = 100 * (all_preds[mask] == c).mean()
        print(f"  {CLASS_NAMES[c]:20s}: {acc:.1f}% ({mask.sum()} samples)")


In [ ]:
# === CONFUSION MATRIX (YouTube-worthy) ===
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(all_true, all_preds)

fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#0a0a1a')

# Custom colormap
colors_base = np.array([
    [10, 10, 30],    # dark blue-black
    [22, 199, 154],  # teal-green
])

im = ax.imshow(cm, cmap='YlGn', aspect='auto')

# Add text annotations
for i in range(4):
    for j in range(4):
        color = 'white' if cm[i, j] > cm.max() * 0.5 else '#16c79a'
        ax.text(j, i, f'{cm[i,j]}', ha='center', va='center',
                fontsize=16, fontweight='bold', color=color)

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_NAMES, color='white', fontsize=11)
ax.set_yticklabels(CLASS_NAMES, color='white', fontsize=11)
ax.set_xlabel('Predicted', color='white', fontsize=13)
ax.set_ylabel('True', color='white', fontsize=13)
ax.set_title('Confusion Matrix — 1D CNN INT8 (PYNQ-Z2 Target)', 
             color='white', fontsize=14, pad=15)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, facecolor='#1a1a2e', bbox_inches='tight')
plt.show()

print(f"\n✅ Step 2 complete. Test accuracy: {test_acc:.2f}%")
print("Saved: best_model.pth, training_curves.png, confusion_matrix.png")
